# G33 - Cloud DataCenters: ETL Excel → SQLite

Este notebook carrega os dados do ficheiro Excel (`g33_Cloud_DataCenters_csv_v2.xlsx`) e povoa a base de dados SQLite (`cloud.db`) com as seguintes tabelas:

| Tabela | Fonte |
|---|---|
| `Provider` | Folha `g33_Cloud_DataCenters` (colunas `provider_id`, `name`, `creation_date`) |
| `Datacenter` | Folha `g33_Cloud_DataCenters` (colunas `datacenter_id`, `title`, `category`) |
| `ServerSpec` | Folha `ServerSpec` |
| `Server` | Folha `g33_Cloud_DataCenters` - apenas linhas com `server_id` não nulo |
| `UsageQuota` | Folha `g33_Cloud_DataCenters` (colunas `provider_id`, `datacenter_id`, `usage_date`, `cost`) |

**Relações entre classes:**
- `Provider` 1 → N `Server`
- `ServerSpec` 1 → N `Server`
- `Provider` 1..* ↔ 1..* `Datacenter` (via `UsageQuota`)

## 1. Imports e Configuração

In [1]:
import pandas as pd
import sqlite3
import os

EXCEL_PATH = 'g33_Cloud_DataCenters_csv_v2.xlsx'
DB_PATH    = 'data/cloud.db'

for f in [EXCEL_PATH, DB_PATH]:
    status = '✅ encontrado' if os.path.exists(f) else '❌ NÃO encontrado'
    print(f'{f}: {status}')

g33_Cloud_DataCenters_csv_v2.xlsx: ✅ encontrado
data/cloud.db: ✅ encontrado


## 2. Leitura do Excel

In [2]:
xl = pd.ExcelFile(EXCEL_PATH)
print('Folhas encontradas:', xl.sheet_names)

df_main = pd.read_excel(xl, sheet_name='g33_Cloud_DataCenters')

df_spec = pd.read_excel(xl, sheet_name='ServerSpec')

print(f'\ng33_Cloud_DataCenters: {df_main.shape[0]} linhas, {df_main.shape[1]} colunas')
print(f'ServerSpec:            {df_spec.shape[0]} linhas, {df_spec.shape[1]} colunas')

Folhas encontradas: ['g33_Cloud_DataCenters', 'ServerSpec']

g33_Cloud_DataCenters: 10772 linhas, 11 colunas
ServerSpec:            40 linhas, 5 colunas


In [3]:
df_main.head()

,provider_id,datacenter_id,usage_date,cost,name,creation_date,title,category,server_id,extra_info,spec_id
0,375,410,2025-09-26,2636,Carpenter-Daniel,2021-03-27,End,fine,119.0,court,13.0
1,70,431,2023-05-26,4322,Huynh-Chase,2022-06-10,Beat,remain,176.0,season,17.0
2,182,667,2023-03-13,417,"Chapman, Hart and Turner",2023-01-30,Edge,mention,NaN,NaN,NaN
3,122,174,2020-05-06,1426,"Stanton, Hahn and Hebert",2022-03-25,How,yes,NaN,NaN,NaN
4,214,871,2024-01-20,1936,"Brown, Kim and Wright",2024-08-21,His,still,NaN,NaN,NaN


In [4]:
df_spec.head()

,spec_id,ram_gb,cpu_cores,operating_system,storage_gb
0,1,512,4,Ubuntu 22.04,1024
1,2,32,8,CentOS 8,256
2,3,512,64,Ubuntu 22.04,4096
3,4,128,4,Ubuntu 22.04,256
4,5,32,8,RHEL 9,4096


## 3. Transformação dos Dados

A folha principal é uma tabela desnormalizada. Vamos extrair cada entidade de forma independente, removendo duplicados.

### 3.1 Provider

In [5]:
df_provider = (
    df_main[['provider_id', 'name', 'creation_date']]
    .drop_duplicates(subset='provider_id')
    .rename(columns={'provider_id': 'id'})
    .sort_values('id')
    .reset_index(drop=True)
)

df_provider['creation_date'] = pd.to_datetime(df_provider['creation_date']).dt.date.astype(str)

print(f'Providers: {len(df_provider)} registos únicos')
df_provider.head()

Providers: 500 registos únicos


,id,name,creation_date
0,1,Branch-Murray,2025-09-21
1,2,Anthony-Potter,2022-09-20
2,3,"Williams, Moyer and Rose",2024-01-06
3,4,Rogers-Gomez,2023-01-12
4,5,"Thompson, Sanchez and Miller",2025-07-24


### 3.2 Datacenter

In [6]:
df_datacenter = (
    df_main[['datacenter_id', 'title', 'category']]
    .drop_duplicates(subset='datacenter_id')
    .rename(columns={'datacenter_id': 'id'})
    .sort_values('id')
    .reset_index(drop=True)
)

print(f'Datacenters: {len(df_datacenter)} registos únicos')
df_datacenter.head()

Datacenters: 1000 registos únicos


,id,title,category
0,1,Evidence,together
1,2,Land,save
2,3,Hold,TV
3,4,Future,evening
4,5,Painting,form


### 3.3 ServerSpec

In [7]:
df_serverspec = (
    df_spec
    .rename(columns={'spec_id': 'id'})
    .sort_values('id')
    .reset_index(drop=True)
)

print(f'ServerSpecs: {len(df_serverspec)} registos')
df_serverspec.head()

ServerSpecs: 40 registos


,id,ram_gb,cpu_cores,operating_system,storage_gb
0,1,512,4,Ubuntu 22.04,1024
1,2,32,8,CentOS 8,256
2,3,512,64,Ubuntu 22.04,4096
3,4,128,4,Ubuntu 22.04,256
4,5,32,8,RHEL 9,4096


### 3.4 Server

> Apenas as linhas com `server_id` não nulo.

In [8]:
df_server = (
    df_main[df_main['server_id'].notna()][['server_id', 'extra_info', 'provider_id', 'spec_id']]
    .drop_duplicates(subset='server_id')
    .rename(columns={'server_id': 'id'})
    .assign(
        id=lambda x: x['id'].astype(int),
        provider_id=lambda x: x['provider_id'].astype(int),
        spec_id=lambda x: x['spec_id'].astype(int)
    )
    .sort_values('id')
    .reset_index(drop=True)
)

print(f'Servers: {len(df_server)} registos únicos')
df_server.head()

Servers: 200 registos únicos


,id,extra_info,provider_id,spec_id
0,1,huge,220,39
1,2,lot,202,21
2,3,usually,52,32
3,4,practice,353,2
4,5,account,50,8


### 3.5 UsageQuota

In [9]:
df_usage = (
    df_main[['provider_id', 'datacenter_id', 'usage_date', 'cost']]
    .copy()
    .reset_index(drop=True)
)

df_usage.insert(0, 'id', range(1, len(df_usage) + 1))

df_usage['usage_date'] = pd.to_datetime(df_usage['usage_date']).dt.date.astype(str)

print(f'UsageQuota: {len(df_usage)} registos')
df_usage.head()

UsageQuota: 10772 registos


,id,provider_id,datacenter_id,usage_date,cost
0,1,375,410,2025-09-26,2636
1,2,70,431,2023-05-26,4322
2,3,182,667,2023-03-13,417
3,4,122,174,2020-05-06,1426
4,5,214,871,2024-01-20,1936


## 4. Resumo da Transformação

In [10]:
resumo = pd.DataFrame({
    'Tabela':   ['Provider', 'Datacenter', 'ServerSpec', 'Server', 'UsageQuota'],
    'Registos': [len(df_provider), len(df_datacenter), len(df_serverspec), len(df_server), len(df_usage)]
})
print(resumo.to_string(index=False))

    Tabela  Registos
  Provider       500
Datacenter      1000
ServerSpec        40
    Server       200
UsageQuota     10772


## 5. Carregamento na Base de Dados SQLite

A ordem de inserção respeita a integridade referencial:
1. `Provider` e `Datacenter` (sem dependências)
2. `ServerSpec` (sem dependências)
3. `Server` (depende de `Provider` e `ServerSpec`)
4. `UsageQuota` (depende de `Provider` e `Datacenter`)

In [11]:
def inserir_tabela(df, tabela, conn):
    """Insere um DataFrame numa tabela SQLite, limpando-a primeiro."""
    cur = conn.cursor()
    cur.execute(f'DELETE FROM {tabela}')
    df.to_sql(tabela, conn, if_exists='append', index=False)
    count = conn.execute(f'SELECT COUNT(*) FROM {tabela}').fetchone()[0]
    print(f'  ✅ {tabela}: {count} registos inseridos')

conn = sqlite3.connect(DB_PATH)

print('A inserir dados na base de dados...')
inserir_tabela(df_provider,   'Provider',   conn)
inserir_tabela(df_datacenter, 'Datacenter', conn)
inserir_tabela(df_serverspec, 'ServerSpec', conn)
inserir_tabela(df_server,     'Server',     conn)
inserir_tabela(df_usage,      'UsageQuota', conn)

conn.commit()
conn.close()
print('\n✅ ETL concluído com sucesso!')

A inserir dados na base de dados...
  ✅ Provider: 500 registos inseridos
  ✅ Datacenter: 1000 registos inseridos
  ✅ ServerSpec: 40 registos inseridos
  ✅ Server: 200 registos inseridos
  ✅ UsageQuota: 10772 registos inseridos

✅ ETL concluído com sucesso!


## 6. Verificação Final

In [12]:
conn = sqlite3.connect(DB_PATH)

for tabela in ['Provider', 'Datacenter', 'ServerSpec', 'Server', 'UsageQuota']:
    count = conn.execute(f'SELECT COUNT(*) FROM {tabela}').fetchone()[0]
    print(f'{tabela:12s}: {count} registos')

conn.close()

Provider    : 500 registos
Datacenter  : 1000 registos
ServerSpec  : 40 registos
Server      : 200 registos
UsageQuota  : 10772 registos


In [13]:
conn = sqlite3.connect(DB_PATH)

print('=== Provider (5 primeiros) ===')
display(pd.read_sql('SELECT * FROM Provider LIMIT 5', conn))

print('\n=== Server (5 primeiros) ===')
display(pd.read_sql('SELECT * FROM Server LIMIT 5', conn))

print('\n=== UsageQuota (5 primeiros) ===')
display(pd.read_sql('SELECT * FROM UsageQuota LIMIT 5', conn))

conn.close()

=== Provider (5 primeiros) ===


,id,name,creation_date
0,1,Branch-Murray,2025-09-21
1,2,Anthony-Potter,2022-09-20
2,3,"Williams, Moyer and Rose",2024-01-06
3,4,Rogers-Gomez,2023-01-12
4,5,"Thompson, Sanchez and Miller",2025-07-24



=== Server (5 primeiros) ===


,id,extra_info,provider_id,spec_id
0,1,huge,220,39
1,2,lot,202,21
2,3,usually,52,32
3,4,practice,353,2
4,5,account,50,8



=== UsageQuota (5 primeiros) ===


,id,provider_id,datacenter_id,usage_date,cost
0,1,375,410,2025-09-26,2636.0
1,2,70,431,2023-05-26,4322.0
2,3,182,667,2023-03-13,417.0
3,4,122,174,2020-05-06,1426.0
4,5,214,871,2024-01-20,1936.0


## 7. Consulta de Exemplo

Custo total por Provider (top 10).

In [14]:
conn = sqlite3.connect(DB_PATH)

query = """
    SELECT p.name, COUNT(u.id) AS total_usos, ROUND(SUM(u.cost), 2) AS custo_total
    FROM UsageQuota u
    JOIN Provider p ON u.provider_id = p.id
    GROUP BY p.id
    ORDER BY custo_total DESC
    LIMIT 10
"""

df_result = pd.read_sql(query, conn)
conn.close()

print('Top 10 Providers por Custo Total:')
display(df_result)

Top 10 Providers por Custo Total:


,name,total_usos,custo_total
0,Brown PLC,81,228558.0
1,Cooke and Sons,81,204231.0
2,Strickland-Taylor,66,203115.0
3,"Mejia, Sullivan and Warren",66,189153.0
4,"Brewer, Bell and Shelton",72,182502.0
5,Kramer LLC,63,160716.0
6,"Flores, Brooks and Dunlap",54,138456.0
7,"Scott, Smith and Ford",46,138286.0
8,Moore-Pena,40,133684.0
9,Williams-Martinez,46,126352.0
